# WorkforceIQ - Layoff Risk Prediction Model
## Phase 2: XGBoost Training + SHAP Explainability

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✅ Libraries loaded successfully!')

## 1. Load Data

In [ ]:
# Load training data
df = pd.read_csv('../data/processed/layoffs_training_data.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nTarget distribution:\n{df['layoff_risk'].value_counts()}")
print(f"\nMissing values:\n{df.isnull().sum()}")

df.head()

## 2. Data Preparation

In [ ]:
# Define features
feature_columns = [
    'revenue', 'revenue_growth', 'operating_income', 'net_income',
    'net_income_change', 'total_debt', 'cash', 'debt_to_cash_ratio',
    'operating_cashflow', 'free_cashflow', 'market_cap', 'employee_count',
    'profit_margin', 'beta', 'pe_ratio', 'revenue_per_employee',
    'operating_margin', 'is_profitable'
]

# Filter to available features
available_features = [col for col in feature_columns if col in df.columns]
print(f"Available features ({len(available_features)}): {available_features}")

# Prepare X and y
X = df[available_features].copy()
y = df['layoff_risk'].copy()

# Handle missing values (fill with median)
for col in X.columns:
    if X[col].isnull().any():
        X[col].fillna(X[col].median(), inplace=True)

print(f"\nFinal X shape: {X.shape}")
print(f"Final y shape: {y.shape}")
print(f"\nClass balance: {y.value_counts(normalize=True).to_dict()}")

## 3. Train-Test Split

In [ ]:
# Split data (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape}, Test set: {X_test.shape}")
print(f"Train class distribution: {y_train.value_counts().to_dict()}")
print(f"Test class distribution: {y_test.value_counts().to_dict()}")

## 4. Train XGBoost Model

In [ ]:
# Calculate scale_pos_weight for imbalanced data
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight: {scale_pos_weight:.2f}")

# Initialize XGBoost
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)

# Train
print("\nTraining model...")
model.fit(X_train, y_train)
print("✅ Model trained!")

## 5. Model Evaluation

In [ ]:
# Predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Metrics
print("CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred, target_names=['No Layoff', 'Layoff']))

roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"\nROC-AUC Score: {roc_auc:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Layoff', 'Layoff'],
            yticklabels=['No Layoff', 'Layoff'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../backend/models/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../backend/models/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Feature Importance

In [ ]:
# XGBoost built-in feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importance.head(15), x='importance', y='feature')
plt.title('Top 15 Features by Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../backend/models/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 10 Features:")
print(feature_importance.head(10))

## 7. SHAP Explainability

In [ ]:
# Initialize SHAP explainer
print("Creating SHAP explainer...")
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
print("✅ SHAP values calculated!")

In [ ]:
# SHAP Summary Plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.savefig('../backend/models/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Feature Importance
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig('../backend/models/shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Example: SHAP Force Plot for a High-Risk Prediction
high_risk_idx = np.where(y_pred_proba > 0.7)[0]
if len(high_risk_idx) > 0:
    idx = high_risk_idx[0]
    print(f"Explaining prediction for test sample {idx}")
    print(f"Predicted probability: {y_pred_proba[idx]:.3f}")
    print(f"Actual label: {y_test.iloc[idx]}")
    
    shap.force_plot(
        explainer.expected_value, 
        shap_values[idx], 
        X_test.iloc[idx],
        matplotlib=True,
        show=False
    )
    plt.tight_layout()
    plt.savefig('../backend/models/shap_force_example.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No high-risk predictions found in test set")

## 8. Save Model & Artifacts

In [ ]:
# Save trained model
model_path = '../backend/models/layoff_predictor.joblib'
joblib.dump(model, model_path)
print(f"✅ Model saved: {model_path}")

# Save SHAP explainer
explainer_path = '../backend/models/shap_explainer.joblib'
joblib.dump(explainer, explainer_path)
print(f"✅ SHAP explainer saved: {explainer_path}")

# Save feature list
feature_list_path = '../backend/models/feature_list.joblib'
joblib.dump(list(X.columns), feature_list_path)
print(f"✅ Feature list saved: {feature_list_path}")

# Save scaler info (if needed)
model_info = {
    'features': list(X.columns),
    'n_features': len(X.columns),
    'roc_auc': roc_auc,
    'feature_importance': feature_importance.to_dict('records')[:10],
    'training_samples': len(X_train),
    'test_samples': len(X_test)
}

import json
with open('../backend/models/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)
print(f"✅ Model info saved: model_info.json")

## 9. Test Prediction Function

In [ ]:
def predict_layoff_risk(features_dict, model, feature_list):
    """
    Predict layoff risk for a company.
    
    Args:
        features_dict: Dictionary with feature values
        model: Trained XGBoost model
        feature_list: List of feature names
    
    Returns:
        Dictionary with prediction results
    """
    # Create DataFrame with correct feature order
    X_pred = pd.DataFrame([features_dict])[feature_list]
    
    # Fill missing with 0 (or could use median)
    X_pred = X_pred.fillna(0)
    
    # Predict
    probability = model.predict_proba(X_pred)[0, 1]
    prediction = model.predict(X_pred)[0]
    
    # Calculate SHAP values
    shap_vals = explainer.shap_values(X_pred)[0]
    
    # Get top contributing features
    feature_contributions = pd.DataFrame({
        'feature': feature_list,
        'value': X_pred.values[0],
        'shap_value': shap_vals
    }).sort_values('shap_value', key=abs, ascending=False).head(5)
    
    # Risk level
    if probability >= 0.7:
        risk_level = "HIGH"
    elif probability >= 0.4:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"
    
    return {
        'risk_probability': float(probability),
        'risk_level': risk_level,
        'prediction': int(prediction),
        'top_factors': feature_contributions.to_dict('records')
    }

# Test with a sample
test_sample = X_test.iloc[0].to_dict()
result = predict_layoff_risk(test_sample, model, list(X.columns))

print("\n=== PREDICTION TEST ===")
print(f"Risk Probability: {result['risk_probability']:.2%}")
print(f"Risk Level: {result['risk_level']}")
print(f"Prediction: {'Layoff' if result['prediction'] == 1 else 'No Layoff'}")
print(f"\nTop Contributing Factors:")
for factor in result['top_factors']:
    print(f"  {factor['feature']}: {factor['value']:.2f} (SHAP: {factor['shap_value']:.3f})")

## ✅ Phase 2 Complete!

### What was accomplished:
1. ✅ Loaded and prepared training data
2. ✅ Trained XGBoost classifier
3. ✅ Evaluated model performance (ROC-AUC, confusion matrix)
4. ✅ Generated feature importance plots
5. ✅ Created SHAP explainer for interpretability
6. ✅ Saved model artifacts (.joblib files)
7. ✅ Tested prediction function

### Files created:
- `backend/models/layoff_predictor.joblib` - Trained model
- `backend/models/shap_explainer.joblib` - SHAP explainer
- `backend/models/feature_list.joblib` - Feature names
- `backend/models/model_info.json` - Model metadata
- `backend/models/*.png` - Visualization plots

### Next: Phase 3 - FastAPI Backend